# ИТИС КФУ · Индивидуальный аналитический проект
## Тема: Tech Layoffs & Hiring Trends (2025–2026) · Аналитический движок и структуры данных
**Студент:** Куриганов Константин Витальевич  
**Группа:** 11-511  
**Дисциплина:** Алгоритмы и структуры данных  

---
### Цель работы (Этап 1: Данные и Хеш-Таблица):
1. Проектирование схемы данных предметной области (`LayoffRecord`) с уникальным строковым ключом `record_id`.
2. Синтетическая генерация масштабируемого датасета (от 1 000 до 100 000+ строк).
3. Разработка собственной хеш-таблицы (`MyHashMap`) без использования стандартного `dict` Python:
   - Разрешение коллизий: метод цепочек (*Separate Chaining*)
   - Хеш-функция: полиномиальный хеш *djb2*
   - Контроль фактора загрузки: $\alpha = \frac{\text{size}}{\text{capacity}} \le 0.75$
   - Динамическое масштабирование (*resize & rehash*) при превышении порога
4. Экспериментальное доказательство эффективности: сравнение времени поиска $O(N)$ (линейный поиск) против $O(1)$ (`MyHashMap`).

## 1. Схема данных и генератор датасета

In [ ]:
from dataclasses import dataclass, asdict
from typing import List, Optional, Tuple, Any
import time

@dataclass
class LayoffRecord:
    record_id: str
    company_name: str
    industry: str
    country: str
    company_size: str
    year: int
    month: int
    date: str  # YYYY-MM-DD
    layoffs_count: int
    workforce_cut_pct: float
    hiring_status: str
    ai_impact_factor: int
    stage: str
    annual_burn_reduction_usd_m: float
    headquarters: str
    source_notes: str = ''

REAL_COMPANIES = [
    {'name': 'Google (Alphabet)', 'industry': 'Search & Cloud', 'country': 'United States', 'size': 'Mega-Corp (5000+)', 'hq': 'Mountain View, CA', 'stage': 'Post-IPO', 'base_cut_pct': 6.0},
    {'name': 'Microsoft', 'industry': 'Cloud & Enterprise SaaS', 'country': 'United States', 'size': 'Mega-Corp (5000+)', 'hq': 'Redmond, WA', 'stage': 'Post-IPO', 'base_cut_pct': 5.0},
    {'name': 'Amazon', 'industry': 'E-Commerce & AWS', 'country': 'United States', 'size': 'Mega-Corp (5000+)', 'hq': 'Seattle, WA', 'stage': 'Post-IPO', 'base_cut_pct': 9.0},
    {'name': 'Meta (Facebook)', 'industry': 'Social Media & VR', 'country': 'United States', 'size': 'Mega-Corp (5000+)', 'hq': 'Menlo Park, CA', 'stage': 'Post-IPO', 'base_cut_pct': 13.0},
    {'name': 'Intel Corp.', 'industry': 'Hardware & Semiconductors', 'country': 'United States', 'size': 'Mega-Corp (5000+)', 'hq': 'Santa Clara, CA', 'stage': 'Post-IPO', 'base_cut_pct': 15.0},
    {'name': 'Tesla Inc.', 'industry': 'Hardware & Semiconductors', 'country': 'United States', 'size': 'Mega-Corp (5000+)', 'hq': 'Austin, TX', 'stage': 'Post-IPO', 'base_cut_pct': 10.2},
    {'name': 'Unity Software', 'industry': 'Gaming & Interactive', 'country': 'United States', 'size': 'Enterprise (501-5000)', 'hq': 'San Francisco, CA', 'stage': 'Post-IPO', 'base_cut_pct': 25.0},
    {'name': 'Shopify', 'industry': 'E-Commerce', 'country': 'Canada', 'size': 'Mega-Corp (5000+)', 'hq': 'Ottawa', 'stage': 'Post-IPO', 'base_cut_pct': 8.5},
    {'name': 'CrowdStrike', 'industry': 'Cybersecurity', 'country': 'United States', 'size': 'Enterprise (501-5000)', 'hq': 'Austin, TX', 'stage': 'Post-IPO', 'base_cut_pct': 5.0},
    {'name': 'Grammarly', 'industry': 'AI & Machine Learning', 'country': 'United States', 'size': 'Enterprise (501-5000)', 'hq': 'San Francisco, CA', 'stage': 'Series D', 'base_cut_pct': 23.0},
]

HIRING_STATUSES = ['Freeze', 'Selective', 'Aggressive AI Roles', 'Restructuring']

class LCG:
    def __init__(self, seed: int = 42):
        self.state = seed
    def random(self) -> float:
        self.state = (self.state * 1664525 + 1013904223) % 4294967296
        return self.state / 4294967296.0

def generate_dataset(count: int = 10000, seed: int = 2026) -> List[LayoffRecord]:
    rng = LCG(seed)
    records = []
    for i in range(count):
        comp = REAL_COMPANIES[int(rng.random() * len(REAL_COMPANIES))]
        year = 2025 if rng.random() > 0.35 else 2026
        month = int(rng.random() * 12) + 1
        day = int(rng.random() * 28) + 1
        date_str = f"{year}-{month:02d}-{day:02d}"
        base_cut = 1200 if 'Mega' in comp['size'] else 350
        layoffs = max(10, int(round(base_cut * (0.4 + rng.random() * 1.8))))
        cut_pct = min(65.0, max(1.5, round((comp['base_cut_pct'] * (0.8 + rng.random() * 0.5)), 1)))
        ai_impact = min(10, max(1, int(round(3 + rng.random() * 7))))
        burn = round((layoffs * 0.18 + rng.random() * 15.0), 1)
        rec = LayoffRecord(
            record_id=f"TL-{year}-{i + 1:06d}",
            company_name=comp['name'] if i < len(REAL_COMPANIES) else f"{comp['name']} #{(i % 15) + 1}",
            industry=comp['industry'],
            country=comp['country'],
            company_size=comp['size'],
            year=year,
            month=month,
            date=date_str,
            layoffs_count=layoffs,
            workforce_cut_pct=cut_pct,
            hiring_status=HIRING_STATUSES[int(rng.random() * len(HIRING_STATUSES))],
            ai_impact_factor=ai_impact,
            stage=comp['stage'],
            annual_burn_reduction_usd_m=burn,
            headquarters=comp['hq'],
            source_notes='AI restructuring' if ai_impact >= 8 else 'Operational efficiency'
        )
        records.append(rec)
    records.sort(key=lambda r: r.date)
    return records

records = generate_dataset(10000)
print(f"Сгенерировано записей: {len(records):,}")
print("Пример первой записи:", records[0])

## 2. Реализация собственной Hash Table (MyHashMap)
Реализуем хеш-таблицу с нуля с открытой адресацией цепочками (*Separate Chaining*), полиномиальным хешем *djb2*, автоматическим динамическим расширением при $\alpha > 0.75$ и вычислением метрик распределения.

In [ ]:
@dataclass
class KeyValuePair:
    key: Any
    value: Any

class MyHashMap:
    MAX_LOAD_FACTOR = 0.75

    def __init__(self, initial_capacity: int = 8):
        self.capacity = initial_capacity
        self.size = 0
        self.buckets = [[] for _ in range(self.capacity)]

    def _hash(self, key: Any) -> int:
        """Полиномиальный хеш djb2"""
        s = str(key)
        h = 5381
        for ch in s:
            h = (((h << 5) + h) + ord(ch)) & 0xFFFFFFFF
        return h

    def _index(self, key: Any) -> int:
        return self._hash(key) % self.capacity

    def get_load_factor(self) -> float:
        return self.size / self.capacity

    def _resize(self) -> None:
        old_buckets = self.buckets
        self.capacity *= 2
        self.buckets = [[] for _ in range(self.capacity)]
        self.size = 0
        for bucket in old_buckets:
            for pair in bucket:
                self.put(pair.key, pair.value)

    def put(self, key: Any, value: Any) -> None:
        idx = self._index(key)
        bucket = self.buckets[idx]
        for pair in bucket:
            if pair.key == key:
                pair.value = value
                return
        if (self.size + 1) / self.capacity > self.MAX_LOAD_FACTOR:
            self._resize()
            idx = self._index(key)
            self.buckets[idx].append(KeyValuePair(key=key, value=value))
        else:
            bucket.append(KeyValuePair(key=key, value=value))
        self.size += 1

    def get(self, key: Any, default: Any = None) -> Any:
        idx = self._index(key)
        for pair in self.buckets[idx]:
            if pair.key == key:
                return pair.value
        return default

    def contains(self, key: Any) -> bool:
        return self.get(key) is not None

    def remove(self, key: Any) -> bool:
        idx = self._index(key)
        bucket = self.buckets[idx]
        for i, pair in enumerate(bucket):
            if pair.key == key:
                bucket.pop(i)
                self.size -= 1
                return True
        return False

    def stats(self) -> dict:
        empty = sum(1 for b in self.buckets if len(b) == 0)
        longest = max((len(b) for b in self.buckets), default=0)
        used = self.capacity - empty
        avg_len = (self.size / used) if used > 0 else 0
        return {
            'size': self.size,
            'capacity': self.capacity,
            'load_factor': self.get_load_factor(),
            'empty_buckets': empty,
            'used_buckets': used,
            'longest_bucket': longest,
            'avg_bucket_length': avg_len,
        }

    def __len__(self): return self.size
    def __getitem__(self, k): return self.get(k)
    def __setitem__(self, k, v): self.put(k, v)

print("Класс MyHashMap успешно скомпилирован.")

## 3. Построение индекса и анализ состояния хеш-таблицы

In [ ]:
# Построение индекса
hmap = MyHashMap(initial_capacity=8)
t0 = time.perf_counter()
for r in records:
    hmap.put(r.record_id, r)
t1 = time.perf_counter()

stats = hmap.stats()
print(f"Индекс построен за {(t1 - t0)*1000:.1f} мс")
print(f"Размер (size): {stats['size']:,}")
print(f"Вместимость (capacity): {stats['capacity']:,}")
print(f"Коэффициент загрузки (load_factor): {stats['load_factor']:.4f}")
print(f"Занятых корзин: {stats['used_buckets']:,} (пустых: {stats['empty_buckets']:,})")
print(f"Самая длинная цепочка коллизий: {stats['longest_bucket']}")
print(f"Средняя длина занятой корзины: {stats['avg_bucket_length']:.2f}")

## 4. Эксперимент (Шаг 8): Сравнение скорости Linear Search vs MyHashMap
Замеряем время выполнения 30 случайных запросов поиска.

In [ ]:
def linear_search(data_list, target_key):
    for item in data_list:
        if item.record_id == target_key:
            return item
    return None

num_queries = 30
step = len(records) // num_queries
test_keys = [records[i].record_id for i in range(0, len(records), step)][:num_queries]

# 1. Linear Search
t0 = time.perf_counter_ns()
for k in test_keys:
    linear_search(records, k)
t1 = time.perf_counter_ns()
linear_ms = (t1 - t0) / 1_000_000.0

# 2. MyHashMap Lookup
t0 = time.perf_counter_ns()
for k in test_keys:
    hmap.get(k)
t1 = time.perf_counter_ns()
hash_ms = (t1 - t0) / 1_000_000.0

ratio = linear_ms / hash_ms if hash_ms > 0 else 0

print(f"{'='*55}")
print(f"Сравнение на {len(records):,} записях ({num_queries} запросов):")
print(f"- Линейный поиск O(N): {linear_ms:.3f} ms ({(linear_ms/num_queries)*1000:.2f} μs/query)")
print(f"- MyHashMap поиск O(1): {hash_ms:.3f} ms ({(hash_ms/num_queries)*1000:.2f} μs/query)")
print(f"ИТОГ: Хеш-таблица быстрее в {ratio:.1f}x раз!")
print(f"{'='*55}")